In [ ]:

# Evaluation of Energy-Efficient Personalised Optimisation on CIFAR-10


!pip -q install pandas

import copy
import random
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, random_split


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


BATCH_SIZE = 64
NUM_CLASSES = 10
NUM_SERVICES = 50
DIRICHLET_ALPHA = 0.5

GLOBAL_ROUNDS = 5
LOCAL_EPOCHS = 10
LR = 1e-3

# energy-aware optimisation hyperparameters
LAMBDA_ENERGY = 0.8            # energy regularisation weight
RHO_DUAL = 0.5                 # dual update step size
PROX_MU = 1e-4                # proximal weight for personalised head

# simple energy proxy scaling
ENERGY_ALPHA = 0.020


transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_full = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)
test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

train_size = 45000
val_size = 5000
train_set, val_set = random_split(train_full, [train_size, val_size])


def extract_labels(dataset):
    labels = []
    for i in range(len(dataset)):
        _, y = dataset[i]
        labels.append(y)
    return np.array(labels)

def dirichlet_partition(dataset, num_clients, alpha=0.5, seed=42):
    rng = np.random.default_rng(seed)
    labels = extract_labels(dataset)
    num_classes = len(np.unique(labels))

    class_indices = [np.where(labels == c)[0] for c in range(num_classes)]
    client_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        idxs = class_indices[c]
        rng.shuffle(idxs)
        proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        split_points = (np.cumsum(proportions) * len(idxs)).astype(int)[:-1]
        splits = np.split(idxs, split_points)
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())

    return client_indices

train_partitions = dirichlet_partition(train_set, NUM_SERVICES, alpha=DIRICHLET_ALPHA, seed=42)
test_partitions = dirichlet_partition(test_set, NUM_SERVICES, alpha=DIRICHLET_ALPHA, seed=123)


@dataclass
class ServiceState:
    energy_budget: float
    alpha: float
    local_steps: int

def make_service_state(idx):
    group = idx % 3
    if group == 0:
        return ServiceState(
            energy_budget=1.00,
            alpha=0.95 * ENERGY_ALPHA,
            local_steps=LOCAL_EPOCHS
        )
    elif group == 1:
        return ServiceState(
            energy_budget=1.10,
            alpha=1.00 * ENERGY_ALPHA,
            local_steps=LOCAL_EPOCHS
        )
    else:
        return ServiceState(
            energy_budget=1.20,
            alpha=1.05 * ENERGY_ALPHA,
            local_steps=LOCAL_EPOCHS
        )


def build_service_loaders():
    service_loaders = []
    for i in range(NUM_SERVICES):
        train_idx = train_partitions[i]
        test_idx = test_partitions[i]

        if len(train_idx) < 20:
            extra = np.random.choice(len(train_set), 50, replace=False).tolist()
            train_idx.extend(extra)
        if len(test_idx) < 10:
            extra = np.random.choice(len(test_set), 20, replace=False).tolist()
            test_idx.extend(extra)

        local_train = Subset(train_set, train_idx)
        local_test = Subset(test_set, test_idx)

        train_loader = DataLoader(local_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        test_loader = DataLoader(local_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        service_loaders.append((train_loader, test_loader))
    return service_loaders

service_loaders = build_service_loaders()


class SharedPersonalCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.personal_head = nn.Linear(64, num_classes)

    def forward_features(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        feat = self.forward_features(x)
        return self.personal_head(feat)


@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    total = 0
    correct = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        total += y.size(0)
        correct += (pred == y).sum().item()
    return 100.0 * correct / max(total, 1)

def estimate_training_energy(model, state):

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return state.alpha * (num_params / 1e6) * state.local_steps

def get_backbone_state(model):
    return copy.deepcopy(model.backbone.state_dict())

def get_head_state(model):
    return copy.deepcopy(model.personal_head.state_dict())

def set_backbone_state(model, state_dict):
    model.backbone.load_state_dict(copy.deepcopy(state_dict), strict=False)

def set_head_state(model, state_dict):
    model.personal_head.load_state_dict(copy.deepcopy(state_dict), strict=False)

def fedavg_state_dicts(state_dicts, weights):
    total = float(sum(weights))
    out = copy.deepcopy(state_dicts[0])
    for k in out.keys():
        out[k] = sum(sd[k] * (w / total) for sd, w in zip(state_dicts, weights))
    return out


def local_train_standard_fl(global_model, train_loader, state, epochs=1, lr=1e-3):
    model = copy.deepcopy(global_model).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

    energy = estimate_training_energy(model, state)
    return model, energy

def local_train_personalised_no_energy(global_model, train_loader, state, epochs=1, lr=1e-3, mu=1e-4):
    model = copy.deepcopy(global_model).to(device)

    backbone_opt = optim.Adam(model.backbone.parameters(), lr=lr)
    head_opt = optim.Adam(model.personal_head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    head_anchor = copy.deepcopy(model.personal_head.state_dict())

    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # update backbone
            backbone_opt.zero_grad()
            head_opt.zero_grad()

            logits = model(x)
            ce = criterion(logits, y)

            prox = 0.0
            for name, p in model.personal_head.named_parameters():
                prox = prox + torch.sum((p - head_anchor[name].to(device)) ** 2)

            loss = ce + 0.5 * mu * prox
            loss.backward()
            backbone_opt.step()
            head_opt.step()

    energy = estimate_training_energy(model, state)
    return model, energy

def local_train_energy_regularised_no_dual(global_model, train_loader, state, epochs=1, lr=1e-3, mu=1e-4, lambda_energy=0.8):
    model = copy.deepcopy(global_model).to(device)

    backbone_opt = optim.Adam(model.backbone.parameters(), lr=lr)
    head_opt = optim.Adam(model.personal_head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    head_anchor = copy.deepcopy(model.personal_head.state_dict())
    energy_term = estimate_training_energy(model, state)

    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            backbone_opt.zero_grad()
            head_opt.zero_grad()

            logits = model(x)
            ce = criterion(logits, y)

            prox = 0.0
            for name, p in model.personal_head.named_parameters():
                prox = prox + torch.sum((p - head_anchor[name].to(device)) ** 2)

            loss = ce + 0.5 * mu * prox + lambda_energy * energy_term
            loss.backward()
            backbone_opt.step()
            head_opt.step()

    energy = estimate_training_energy(model, state)
    return model, energy

def local_train_orchnas(global_model, train_loader, state, dual_eta, epochs=1, lr=1e-3, mu=1e-4, rho=0.5):
    model = copy.deepcopy(global_model).to(device)

    backbone_opt = optim.Adam(model.backbone.parameters(), lr=lr)
    head_opt = optim.Adam(model.personal_head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    head_anchor = copy.deepcopy(model.personal_head.state_dict())
    energy_term = estimate_training_energy(model, state)

    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            backbone_opt.zero_grad()
            head_opt.zero_grad()

            logits = model(x)
            ce = criterion(logits, y)

            prox = 0.0
            for name, p in model.personal_head.named_parameters():
                prox = prox + torch.sum((p - head_anchor[name].to(device)) ** 2)

            loss = ce + 0.5 * mu * prox + dual_eta * energy_term
            loss.backward()
            backbone_opt.step()
            head_opt.step()

    energy = estimate_training_energy(model, state)
    new_dual_eta = max(0.0, dual_eta + rho * (energy - state.energy_budget))
    return model, energy, new_dual_eta


def run_method(method_name):
    print("\n" + "=" * 70)
    print(f"Running method: {method_name}")
    print("=" * 70)

    global_model = SharedPersonalCNN(num_classes=NUM_CLASSES).to(device)
    global_backbone_state = get_backbone_state(global_model)
    global_head_state = get_head_state(global_model)

    # for OrchNAS only
    dual_vars = [0.0 for _ in range(NUM_SERVICES)]

    for rnd in range(1, GLOBAL_ROUNDS + 1):
        local_backbones = []
        local_heads = []
        local_weights = []

        round_accs = []
        round_energies = []
        round_violations = 0

        for service_id in range(NUM_SERVICES):
            state = make_service_state(service_id)
            train_loader, test_loader = service_loaders[service_id]

            local_model = SharedPersonalCNN(num_classes=NUM_CLASSES).to(device)
            set_backbone_state(local_model, global_backbone_state)
            set_head_state(local_model, global_head_state)

            if method_name == "Standard Federated Optimisation":
                trained_model, energy = local_train_standard_fl(
                    local_model, train_loader, state, epochs=LOCAL_EPOCHS, lr=LR
                )

            elif method_name == "Personalised Training without Energy Regularisation":
                trained_model, energy = local_train_personalised_no_energy(
                    local_model, train_loader, state, epochs=LOCAL_EPOCHS, lr=LR, mu=PROX_MU
                )

            elif method_name == "Energy-Regularised Training without Dual Updates":
                trained_model, energy = local_train_energy_regularised_no_dual(
                    local_model, train_loader, state, epochs=LOCAL_EPOCHS, lr=LR,
                    mu=PROX_MU, lambda_energy=LAMBDA_ENERGY
                )

            elif method_name == "OrchNAS":
                trained_model, energy, dual_vars[service_id] = local_train_orchnas(
                    local_model, train_loader, state, dual_eta=dual_vars[service_id],
                    epochs=LOCAL_EPOCHS, lr=LR, mu=PROX_MU, rho=RHO_DUAL
                )
            else:
                raise ValueError("Unknown method.")

            acc = evaluate_accuracy(trained_model, test_loader)

            if energy > state.energy_budget:
                round_violations += 1

            round_accs.append(acc)
            round_energies.append(energy)

            local_backbones.append(get_backbone_state(trained_model))
            local_heads.append(get_head_state(trained_model))
            local_weights.append(len(train_loader.dataset))

        global_backbone_state = fedavg_state_dicts(local_backbones, local_weights)
        global_head_state = fedavg_state_dicts(local_heads, local_weights)

        print(
            f"Round {rnd:02d} | "
            f"Avg Acc={np.mean(round_accs):.2f}% | "
            f"Worst Acc={np.min(round_accs):.2f}% | "
            f"Violation Rate={100.0 * round_violations / NUM_SERVICES:.2f}% | "
            f"Avg Energy={np.mean(round_energies):.4f} J"
        )

    # final evaluation after last round
    final_accs = []
    final_energies = []
    final_violations = 0

    for service_id in range(NUM_SERVICES):
        state = make_service_state(service_id)
        _, test_loader = service_loaders[service_id]

        eval_model = SharedPersonalCNN(num_classes=NUM_CLASSES).to(device)
        set_backbone_state(eval_model, global_backbone_state)
        set_head_state(eval_model, global_head_state)

        acc = evaluate_accuracy(eval_model, test_loader)
        energy = estimate_training_energy(eval_model, state)

        final_accs.append(acc)
        final_energies.append(energy)

        if energy > state.energy_budget:
            final_violations += 1

    result = {
        "Method": method_name,
        "Average_Personalised_Accuracy (%)": round(float(np.mean(final_accs)), 2),
        "Worst_Service_Accuracy (%)": round(float(np.min(final_accs)), 2),
        "Energy_Violation_Rate (%)": round(100.0 * final_violations / NUM_SERVICES, 2),
        "Average_Training_Energy (J)": round(float(np.mean(final_energies)), 4),
    }

    return result


start_time = time.time()

methods = [
    "Standard Federated Optimisation",
    "Personalised Training without Energy Regularisation",
    "Energy-Regularised Training without Dual Updates",
    "OrchNAS"
]

results = []
for method in methods:
    result = run_method(method)
    results.append(result)

end_time = time.time()

results_df = pd.DataFrame(results)
csv_path = "orchnas_personalised_optimisation_results.csv"
results_df.to_csv(csv_path, index=False)
